# Silver - CETIC 2025

## Objetivo

Transformar os microdados brutos da TIC Domicílios 2025 em uma
estrutura analítica padronizada para o Radar Geracional de Conteúdo.

## Transformações

- seleção das variáveis relevantes;
- tratamento da faixa etária;
- conversão do peso amostral;
- padronização das respostas;
- transformação das categorias de colunas para linhas;
- identificação de respostas válidas e inválidas;
- padronização da nomenclatura das categorias.

## Categorias utilizadas no MVP

- Notícias
- Esportes
- Música
- Humor
- Games

In [0]:
from pyspark.sql import functions as F

In [0]:
df_cetic_bronze = spark.table(
    "workspace.mvp_bronze.cetic_individuos_2025"
)

print("Linhas Bronze:", df_cetic_bronze.count())
print("Colunas Bronze:", len(df_cetic_bronze.columns))

In [0]:
campos_cetic = [
    "QUEST",
    "FAIXA_ETARIA",
    "PESO",
    "TC4B_A",
    "TC4B_B",
    "TC4B_C",
    "TC4B_D",
    "TC4B_F",
    "TC4B_G",
    "TC4B_H",
    "TC4B_I"
]

for campo in campos_cetic:
    print(
        campo,
        "->",
        "OK" if campo in df_cetic_bronze.columns
        else "NÃO ENCONTRADO"
    )

In [0]:
total_pessoas = df_cetic_bronze.count()

quest_distintos = (
    df_cetic_bronze
    .select("QUEST")
    .distinct()
    .count()
)

quest_nulos = (
    df_cetic_bronze
    .filter(F.col("QUEST").isNull())
    .count()
)

print("Total de pessoas:", total_pessoas)
print("QUEST distintos:", quest_distintos)
print("QUEST nulos:", quest_nulos)

In [0]:
df_cetic_base = (
    df_cetic_bronze
    .select(
        "QUEST",
        "FAIXA_ETARIA",
        "PESO",
        "TC4B_A",
        "TC4B_B",
        "TC4B_C",
        "TC4B_D",
        "TC4B_F",
        "TC4B_G",
        "TC4B_H",
        "TC4B_I"
    )
)

print(
    "Colunas selecionadas:",
    len(df_cetic_base.columns)
)

print(
    df_cetic_base.columns
)

In [0]:
df_cetic_tipado = (
    df_cetic_base
    .withColumn(
        "faixa_etaria_codigo",
        F.col("FAIXA_ETARIA").cast("int")
    )
    .withColumn(
        "peso",
        F.regexp_replace(
            F.col("PESO"),
            ",",
            "."
        ).cast("double")
    )
)

In [0]:
display(
    df_cetic_tipado
    .select(
        "QUEST",
        "FAIXA_ETARIA",
        "faixa_etaria_codigo",
        "PESO",
        "peso"
    )
    .limit(20)
)

In [0]:
df_cetic_tipado = (
    df_cetic_tipado
    .withColumn(
        "faixa_etaria",
        F.when(
            F.col("faixa_etaria_codigo") == 1,
            "10-15"
        )
        .when(
            F.col("faixa_etaria_codigo") == 2,
            "16-24"
        )
        .when(
            F.col("faixa_etaria_codigo") == 3,
            "25-34"
        )
        .when(
            F.col("faixa_etaria_codigo") == 4,
            "35-44"
        )
        .when(
            F.col("faixa_etaria_codigo") == 5,
            "45-59"
        )
        .when(
            F.col("faixa_etaria_codigo") == 6,
            "60+"
        )
        .otherwise("Não informado")
    )
)

In [0]:
display(
    df_cetic_tipado
    .groupBy(
        "faixa_etaria_codigo",
        "faixa_etaria"
    )
    .count()
    .orderBy("faixa_etaria_codigo")
)

In [0]:
for campo in [
    "TC4B_A",
    "TC4B_B",
    "TC4B_C",
    "TC4B_D",
    "TC4B_F",
    "TC4B_G",
    "TC4B_H",
    "TC4B_I"
]:
    print(
        campo,
        "->",
        "OK" if campo in df_cetic_tipado.columns
        else "NÃO ENCONTRADO"
    )

In [0]:
df_cetic_long = (
    df_cetic_tipado
    .selectExpr(
        "QUEST as id_respondente",
        "faixa_etaria_codigo",
        "faixa_etaria",
        "peso",
        """
        stack(
            8,
            'Notícias', TC4B_A,
            'Esportes', TC4B_B,
            'Música', TC4B_C,
            'Humor', TC4B_D,
            'Animações', TC4B_F,
            'Games', TC4B_G,
            'Tutoriais / Educação', TC4B_H,
            'Influenciadores', TC4B_I
        ) as (categoria, resposta_original)
        """
    )
)

In [0]:
display(
    df_cetic_long.limit(24)
)

In [0]:
print(
    "Linhas após stack(8):",
    df_cetic_long.count()
)

print(
    "Respondentes distintos:",
    df_cetic_long
    .select("id_respondente")
    .distinct()
    .count()
)

In [0]:
display(
    df_cetic_long
    .groupBy("resposta_original")
    .count()
    .orderBy("resposta_original")
)

In [0]:
df_cetic_long = (
    df_cetic_long
    .withColumn(
        "resposta_codigo",
        F.expr(
            "try_cast(resposta_original as int)"
        )
    )
)

In [0]:
df_cetic_long = (
    df_cetic_long
    .withColumn(
        "resposta_valida",
        F.col("resposta_codigo").isin(
            1,
            2
        )
    )
    .withColumn(
        "resposta",
        F.when(
            F.col("resposta_codigo") == 1,
            1
        )
        .when(
            F.col("resposta_codigo") == 2,
            0
        )
        .otherwise(
            F.lit(None).cast("int")
        )
    )
    .withColumn(
        "status_resposta",
        F.when(
            F.col("resposta_codigo") == 1,
            "Sim"
        )
        .when(
            F.col("resposta_codigo") == 2,
            "Não"
        )
        .when(
            F.col("resposta_codigo") == 97,
            "Não sabe"
        )
        .when(
            F.col("resposta_codigo") == 98,
            "Não respondeu"
        )
        .when(
            F.col("resposta_codigo") == 99,
            "Não se aplica"
        )
        .otherwise(
            "Código não mapeado"
        )
    )
)

In [0]:
display(
    df_cetic_long
    .groupBy(
        "resposta_codigo",
        "status_resposta",
        "resposta_valida"
    )
    .count()
    .orderBy("resposta_codigo")
)

In [0]:
df_cetic_silver = (
    df_cetic_long
    .withColumn(
        "ano_referencia",
        F.lit(2025)
    )
    .withColumn(
        "fonte",
        F.lit(
            "CETIC.br - TIC Domicílios 2025"
        )
    )
    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )
    .select(
        "ano_referencia",
        "id_respondente",
        "faixa_etaria_codigo",
        "faixa_etaria",
        "categoria",
        "resposta_original",
        "resposta_codigo",
        "resposta",
        "resposta_valida",
        "status_resposta",
        "peso",
        "fonte",
        "_data_processamento"
    )
)

In [0]:
display(
    df_cetic_silver.limit(30)
)

In [0]:
print(
    "Total de linhas:",
    df_cetic_silver.count()
)

print(
    "Respondentes distintos:",
    df_cetic_silver
    .select("id_respondente")
    .distinct()
    .count()
)

print(
    "Categorias distintas:",
    df_cetic_silver
    .select("categoria")
    .distinct()
    .count()
)

In [0]:
display(
    df_cetic_silver
    .groupBy("categoria")
    .count()
    .orderBy("categoria")
)

In [0]:
duplicados_cetic = (
    df_cetic_silver
    .groupBy(
        "id_respondente",
        "categoria"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

print(
    "Duplicidades id_respondente + categoria:",
    duplicados_cetic.count()
)

In [0]:
display(
    df_cetic_silver
    .agg(
        F.count("*").alias("registros"),

        F.sum(
            F.when(
                F.col("peso").isNull(),
                1
            ).otherwise(0)
        ).alias("peso_nulo"),

        F.min("peso").alias("peso_minimo"),

        F.max("peso").alias("peso_maximo")
    )
)

In [0]:
(
    df_cetic_silver.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        "workspace.mvp_silver.consumo_digital_cetic_2025"
    )
)

In [0]:
df_cetic_check = spark.table(
    "workspace.mvp_silver.consumo_digital_cetic_2025"
)

print(
    "Linhas gravadas:",
    df_cetic_check.count()
)

print(
    "Respondentes gravados:",
    df_cetic_check
    .select("id_respondente")
    .distinct()
    .count()
)

print(
    "Categorias gravadas:",
    df_cetic_check
    .select("categoria")
    .distinct()
    .count()
)

In [0]:
%sql

SHOW TABLES IN workspace.mvp_silver;

## Resultado da Silver - CETIC

A base TIC Domicílios 2025 foi convertida para uma estrutura
analítica em formato longo.

Foram processados:

- 24.535 respondentes;
- 8 categorias de conteúdo;
- 196.280 registros analíticos.

As categorias utilizadas são:

- Notícias
- Esportes
- Música
- Humor
- Animações
- Games
- Tutoriais / Educação
- Influenciadores

O identificador `QUEST` foi preservado como `id_respondente`,
permitindo validar a granularidade da base pela combinação:

`id_respondente + categoria`

Os códigos originais de resposta foram preservados para
rastreabilidade.

Tabela final:

`workspace.mvp_silver.consumo_digital_cetic_2025`